In [1]:
# install dependencies
# !pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_adaptive.git
# !pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_t5.git
# !pip install -q pyterrier_pisa
# !pip install python-terrier
# !pip install --upgrade git+https://github.com/terrier-org/pyterrier.git#egg=python-terrier
# !pip install --upgrade git+https://github.com/terrierteam/pyterrier_dr.git
# !pip install faiss-cpu 

In [2]:
# !pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_doc2query.git
# !wget https://git.uwaterloo.ca/jimmylin/doc2query-data/raw/master/T5-passage/t5-base.zip
# !unzip t5-base.zip
# install dependencies
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_adaptive.git
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_t5.git
!pip install -q pyterrier_pisa

DEPRECATION: pytorch-lightning 1.5.10 has a non-standard dependency specifier torch>=1.7.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: pytorch-lightning 1.5.10 has a non-standard dependency specifier torch>=1.7.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: pytorch-lightning 1.5.10 has a non-standard dependency specifier torch>=1.7.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the aut

In [1]:
# imports
import pyterrier as pt
if not pt.started():
    pt.init()
from pyterrier.measures import *
from pyterrier_adaptive import GAR, CorpusGraph
from pyterrier_t5 import MonoT5ReRanker
from pyterrier_pisa import PisaIndex
from pyterrier_dr import TctColBert, FlexIndex
from pyterrier_dr.flex import FlexIndex

/home/peppe/anaconda3/envs/GNRR/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTerrier 0.10.0 has loaded Terrier 5.8 (built by craigm on 2023-11-01 18:05) and terrier-helper 0.0.8

No etc/terrier.properties, using terrier.default.properties for bootstrap configuration.


In [2]:
from pyterrier_dr import TctColBert

dataset_name = 'irds:cord19/trec-covid'

dataset = pt.get_dataset(dataset_name)

In [3]:
transformer = TctColBert('castorini/tct_colbert-v2-hnp-msmarco')
index_flex = FlexIndex('corpus_index/corpusgraph_k8')
index = pt.IndexRef.of("./cord19-index")

# indexing_pipeline = model >> index
# indexing_pipeline.index(dataset.get_corpus_iter())
# bm25 = pt.BatchRetrieve(index, wmodel="BM25")

In [4]:
retriever = pt.BatchRetrieve(index, wmodel="BM25", metadata=["docno"]) % 100


12:40:51.188 [main] WARN org.terrier.structures.FSADocumentIndex - This index has fields, but FSADocumentIndex is used (which stores fields lengths on disk); If using field-based models such as BM25F, change to index.document.class in the index  properties file to FSAFieldDocumentIndex or FSADocumentIndexInMemFields to support efficient retrieval. If you don't use (e.g.) BM25F, this warning can be ignored


In [5]:
scorer = MonoT5ReRanker(verbose=False, batch_size=16, text_field = 'abstract')
graph = index_flex
GAR_ = retriever >> GAR(scorer, graph)

/home/peppe/anaconda3/envs/GNRR/lib/python3.8/site-packages/transformers/models/t5/tokenization_t5.py:240: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-base automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer with `model_max_length` set to your preferred value.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If

In [6]:
from pyterrier.measures import *

pt.Experiment(
    [retriever, retriever >> scorer, retriever >> GAR_],
    dataset.get_topics('description'),
    dataset.get_qrels(),
    eval_metrics=[nDCG, MAP(rel=2), R(rel=2)@1000])

KeyError: 'abstract'

In [ ]:
# Create required components
dataset = pt.get_dataset('irds:msmarco-passage')
doc2query = Doc2Query(out_attr="exp_terms", batch_size=8)
index_output_path = './'

K = 16

index_file = "indices_doc2query." + str(K)
index = PisaIndex(index_output_path + index_file)
index_pipeline = doc2query >> pt.apply.text(lambda r: f'{r["text"]} {r["exp_terms"]}') >> index


# dataset = pt.get_dataset('irds:msmarco-passage')

# Build the index needed for BM25 retrieval (if it doesn't already exist)
# idx = PisaIndex('msmarco-passage.pisa', threads=45) # adjust for your resources
# if not idx.built():
#     idx.index(dataset.get_corpus_iter())

# Build the corpus graph
K = 16 # number of nearest neighbours
# graph16 = CorpusGraph.from_retriever(
#     idx.bm25(num_results=K+1), # K+1 needed because retriever will return original document
#     dataset.get_corpus_iter(),
#     'msmarco-passage.gbm25.16',
#     k=K)


idx = index_pipeline.index(dataset.get_corpus_iter())


In [3]:
index_output_path = '/content/drive/MyDrive/GNN_IR/corpus_graph_building/'
name = "file_test." + str(K)


graph16 = CorpusGraph.from_retriever(
    doc2query, # K+1 needed because retriever will return original document
    index_pipeline.index(dataset.get_corpus_iter()),
    index_output_path + name,
    k=K)

[INFO] [starting] fixing encoding                                     
                                                                      
                                      [INFO] [error] fixing encoding: [02:41] [1.51GB] [9.33MB/s]
msmarco-passage documents:   0%|          | 0/8841823 [02:41<?, ?it/s]


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>                                                                                      │
│                                                                                                  │
│    4                                                                                             │
│    5 graph16 = CorpusGraph.from_retriever(                                                       │
│    6 │   doc2query, # K+1 needed because retriever will return original document                 │
│ ❱  7 │   index_pipeline.index(dataset.get_corpus_iter()),                                        │
│    8 │   index_output_path + name,                                                               │
│    9 │   k=K)                                                                                    │
│   10                                                                                             │
│                                                                                                  │
│ /home/peppe/anaconda3/envs/my_env/lib/python3.10/site-packages/pyterrier/ops.py:331 in index     │
│                                                                                                  │
│   328 │   │   │   │   batch_df = prev_transformer.transform_iter(batch)                          │
│   329 │   │   │   │   for row in batch_df.itertuples(index=False):                               │
│   330 │   │   │   │   │   yield row._asdict()                                                    │
│ ❱ 331 │   │   return last_transformer.index(gen())                                               │
│   332 │                                                                                          │
│   333 │   def transform(self, topics):                                                           │
│   334 │   │   for m in self.models:                                                              │
│                                                                                                  │
│ /home/peppe/anaconda3/envs/my_env/lib/python3.10/site-packages/pyterrier_pisa/__init__.py:134 in │
│ index                                                                                            │
│                                                                                                  │
│   131                                                                                            │
│   132   def index(self, it):                                                                     │
│   133 │   it = more_itertools.peekable(it)                                                       │
│ ❱ 134 │   first_doc = it.peek()                                                                  │
│   135 │   text_field = self.text_field                                                           │
│   136 │   if text_field is None: # infer the text field                                          │
│   137 │     dict_field = [k for k, v in sorted(first_doc.items()) if k.endswith('toks') and is   │
│                                                                                                  │
│ /home/peppe/anaconda3/envs/my_env/lib/python3.10/site-packages/more_itertools/more.py:340 in     │
│ peek                                                                                             │
│                                                                                                  │
│    337 │   │   """                                                                               │
│    338 │   │   if not self._cache:                                                               │
│    339 │   │   │   try:                                                                          │
│ ❱  340 │   │   │   │   self._cache.append(next(self._it))                                        │
│    341 │   │   │   except StopIteration:                   

In [ ]:
retriever = PisaIndex.from_dataset('msmarco_passage').bm25()

In [ ]:
scorer = pt.text.get_text(dataset, 'text') >> MonoT5ReRanker(verbose=False, batch_size=16)

In [ ]:
graph = CorpusGraph.from_dataset('msmarco_passage', 'corpusgraph_bm25').to_limit_k(8)

In [ ]:
graph = CorpusGraph.from_dataset('msmarco_passage', 'corpusgraph_bm25_k16').to_limit_k(8)

In [ ]:
print(f"The type of the object is: {type(graph)}")

In [ ]:
# A simple example
pipeline = retriever >> GAR(scorer, graph) >> pt.text.get_text(dataset, 'text')

In [ ]:
# pipeline.search('clustering hypothesis information retrieval')

In [ ]:
# # Create required components
# dataset = pt.get_dataset('irds:msmarco-passage')
# retriever = PisaIndex.from_dataset('msmarco_passage').bm25()
# scorer = pt.text.get_text(dataset, 'text') >> MonoT5ReRanker(verbose=False, batch_size=16)
# graph = CorpusGraph.from_dataset('msmarco_passage', 'corpusgraph_bm25_k16').to_limit_k(8)

In [ ]:
# # A simple example
# pipeline = retriever >> GAR(scorer, graph) >> pt.text.get_text(dataset, 'text')

# pipeline.search('clustering hypothesis information retrieval')

In [ ]:
# dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
# pt.Experiment(
#     [retriever, retriever >> scorer, retriever >> GAR(scorer, graph)],
#     dataset.get_topics(),
#     dataset.get_qrels(),
#     [nDCG, MAP(rel=2), R(rel=2)@1000],
#     names=['bm25', 'bm25 >> monot5', 'bm25 >> GAR(monot5)']
# )

In [ ]:
import os
os.environ['IR_DATASETS_HOME'] = '/content/drive/MyDrive/.ir_datasets'
print("Environment variable set to:", os.environ['IR_DATASETS_HOME'])

### Attempt 1

In [ ]:
index_output_path = '/content/drive/MyDrive/GNN_IR/corpus_graph_building/'

In [ ]:
from pyterrier_doc2query import Doc2Query
doc2query = Doc2Query(out_attr="exp_terms", batch_size=8)

In [ ]:
dataset = pt.get_dataset('irds:msmarco-passage')

In [ ]:
index = PisaIndex(index_output_path + 'msmarco-passage-doc2query-pisa')

In [ ]:
index_pipeline = doc2query >> pt.apply.text(lambda r: f'{r["text"]} {r["exp_terms"]}') >> index

In [ ]:
index_pipeline.index(dataset.get_corpus_iter())

In [ ]:
# doc2query = Doc2Query(out_attr="exp_terms", batch_size=8)
# dataset = pt.get_dataset('irds:msmarco-passage')
# index = PisaIndex(index_output_path + '/msmarco-passage-doc2query-pisa')
# index_pipeline = doc2query >> pt.apply.text(lambda r: f'{r["text"]} {r["exp_terms"]}') >> index
# index_pipeline.index(dataset.get_corpus_iter())

### Costruzione grafo a latere

In [ ]:
from pyterrier_adaptive import CorpusGraph
from pyterrier_pisa import PisaIndex

In [ ]:
dataset = pt.get_dataset('irds:msmarco-passage')

In [ ]:
# Build the index needed for BM25 retrieval (if it doesn't already exist)
idx = PisaIndex('msmarco-passage.pisa', threads=45) # adjust for your resources
if not idx.built():
    idx.index(dataset.get_corpus_iter())

In [ ]:
import os
import shutil
from google.colab import files

def zip_and_download_folder(folder_path, output_name):

    # Ensure the folder exists
    if not os.path.exists(folder_path):
        raise ValueError("The specified folder does not exist.")

    # Create a zip archive of the folder
    shutil.make_archive(output_name, 'zip', folder_path)

    # Download the zip file
    files.download(output_name + '.zip')

folder_to_zip = '/content/msmarco-passage.pisa'
output_zip_name = 'msmarco-passage.pisa_index'
zip_and_download_folder(folder_to_zip, output_zip_name)